# BCS714A Multimodal Product Classification (Assignment 2)
## Dual-Branch Multimodal Fusion Network (Vision + NLP)

**Course:** BCS714A - Deep Learning (Activity-Based Learning)  
**Kaggle Competition:** [BCS714A Multimodal Fusion](https://www.kaggle.com/t/08d5b24fd37f413c99c47d5fe82468bc)  
**Leaderboard Metric:** Balanced Accuracy (Macro-average recall across 140 classes)  

### Architectural Rules & Constraints:
1. **Parameter Cap:** Total model parameters (trainable + frozen) must be strictly **< 15,000,000 (15 Million)**.
2. **No Pre-trained VLMs:** Pre-trained Vision-Language Models (e.g. CLIP, BLIP, LLaVA) are strictly prohibited.
3. **Vision Branch (Module 3):** Lightweight CNN backbone (`MobileNetV2` with 1,280 features or `ResNet-18` with 512 features).
4. **NLP Branch (Modules 4 & 5):** Word Embeddings + Bidirectional Recurrent Neural Network (`GRU` / `LSTM`) with temporal pooling (256 features).
5. **Fusion Layer:** Manual tensor concatenation (`torch.cat([img_feat, text_feat], dim=1)`) followed by an MLP classifier with Dropout and BatchNorm.
6. **Metric Alignment:** Balanced class weighting in Cross-Entropy Loss to handle severe class imbalance across 140 categories.
7. **Verification:** Model summary with exact parameter audit printed and asserted `< 15,000,000`.

In [ ]:
# ==============================================================================
# 1. IMPORTS, REPRODUCIBILITY & DEVICE SETUP
# ==============================================================================
import os
import re
import gc
import time
import random
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

# Set random seeds for exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")


In [ ]:
# ==============================================================================
# 2. PATHS & HYPERPARAMETERS (Auto-Detects Kaggle vs Local)
# ==============================================================================
KAGGLE_DATA_DIR = '/kaggle/input/competitions/bcs-714-a-multimodal-fusion/data/data/'
KAGGLE_TRAIN_CSV = '/kaggle/input/competitions/bcs-714-a-multimodal-fusion/train.csv'
KAGGLE_TEST_CSV = '/kaggle/input/competitions/bcs-714-a-multimodal-fusion/test.csv'

LOCAL_DATA_DIR = r'C:\Users\dheem\Downloads\bcs-714-a-multimodal-fusion\data\data'
LOCAL_TRAIN_CSV = r'C:\Users\dheem\Downloads\bcs-714-a-multimodal-fusion\train.csv'
LOCAL_TEST_CSV = r'C:\Users\dheem\Downloads\bcs-714-a-multimodal-fusion\test.csv'

if os.path.exists(KAGGLE_TRAIN_CSV):
    DATA_DIR = KAGGLE_DATA_DIR
    TRAIN_CSV = KAGGLE_TRAIN_CSV
    TEST_CSV = KAGGLE_TEST_CSV
    print("Environment: Kaggle detected.")
elif os.path.exists(LOCAL_TRAIN_CSV):
    DATA_DIR = LOCAL_DATA_DIR
    TRAIN_CSV = LOCAL_TRAIN_CSV
    TEST_CSV = LOCAL_TEST_CSV
    print("Environment: Local machine detected.")
else:
    DATA_DIR = 'data/data/'
    TRAIN_CSV = 'train.csv'
    TEST_CSV = 'test.csv'
    print("Environment: Working directory fallback.")

# Architecture and training hyperparameters
BATCH_SIZE = 64
EPOCHS = 5
IMAGE_SIZE = (128, 128)
MAX_TEXT_LEN = 60
VOCAB_SIZE = 10000
EMBED_DIM = 128
HIDDEN_DIM = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
VISION_BACKBONE = 'mobilenet_v2'  # 'mobilenet_v2' (~4.7M total params) or 'resnet18' (~13.2M params)


In [ ]:
# ==============================================================================
# 3. DATA PREPROCESSING, VOCABULARY & LABEL ENCODING
# ==============================================================================
print("Loading dataset...")
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print(f"Train samples: {len(train_df):,}, Test samples: {len(test_df):,}")

# 1. Label Encoders (140 classes)
unique_categories = sorted(train_df['category'].unique())
num_classes = len(unique_categories)
cat2idx = {cat: idx for idx, cat in enumerate(unique_categories)}
idx2cat = {idx: cat for cat, idx in cat2idx.items()}
print(f"Target classes count: {num_classes}")

# 2. Text Preprocessing: Combine display name + description
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train_df['combined_text'] = (
    train_df['display name'].fillna('').astype(str) + " " + train_df['description'].fillna('').astype(str)
)
test_df['combined_text'] = (
    test_df['display name'].fillna('').astype(str) + " " + test_df['description'].fillna('').astype(str)
)

# 3. Build Text Vocabulary
counter = Counter()
for txt in train_df['combined_text']:
    counter.update(clean_text(txt).split())

word2idx = {'<PAD>': 0, '<UNK>': 1}
for idx, (word, _) in enumerate(counter.most_common(VOCAB_SIZE - 2), start=2):
    word2idx[word] = idx

def tokenize_and_pad(text, word_dict, max_len=MAX_TEXT_LEN):
    tokens = clean_text(text).split()
    indices = [word_dict.get(w, 1) for w in tokens]
    if len(indices) > max_len:
        indices = indices[:max_len]
    else:
        indices += [0] * (max_len - len(indices))
    return torch.tensor(indices, dtype=torch.long)

print(f"Vocabulary ready with {len(word2idx):,} entries.")


In [ ]:
# ==============================================================================
# 4. PYTORCH DATASET & DATA AUGMENTATION TRANSFORMS
# ==============================================================================
class MultimodalDataset(Dataset):
    def __init__(self, df, img_dir, word_dict, max_text_len, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.word_dict = word_dict
        self.max_text_len = max_text_len
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        
        # 1. Image Modality
        img_name = os.path.join(self.img_dir, str(row['image']))
        try:
            image = Image.open(img_name).convert('RGB')
        except Exception:
            image = Image.new('RGB', IMAGE_SIZE, color=(128, 128, 128))
            
        if self.transform:
            image = self.transform(image)
            
        # 2. Text Modality
        text_tensor = tokenize_and_pad(row['combined_text'], self.word_dict, self.max_text_len)
        
        # 3. Output
        if self.is_test:
            item_id = int(row['id'])
            return image, text_tensor, item_id
        else:
            label = cat2idx[row['category']]
            return image, text_tensor, torch.tensor(label, dtype=torch.long)

# Data Augmentation for training, deterministic resizing for evaluation
norm_mean = [0.485, 0.456, 0.406]
norm_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((int(IMAGE_SIZE[0] * 1.15), int(IMAGE_SIZE[1] * 1.15))),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm_mean, std=norm_std)
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=norm_mean, std=norm_std)
])

# Stratified Split preserving rare classes (for Balanced Accuracy)
counts = train_df['category'].value_counts()
multi_classes = counts[counts >= 2].index
single_classes = counts[counts == 1].index

df_multi = train_df[train_df['category'].isin(multi_classes)]
df_single = train_df[train_df['category'].isin(single_classes)]

train_split, val_split = train_test_split(
    df_multi, test_size=0.15, stratify=df_multi['category'], random_state=42
)
train_split = pd.concat([train_split, df_single], ignore_index=True)
val_split = val_split.reset_index(drop=True)

train_dataset = MultimodalDataset(train_split, DATA_DIR, word2idx, MAX_TEXT_LEN, transform=train_transform)
val_dataset = MultimodalDataset(val_split, DATA_DIR, word2idx, MAX_TEXT_LEN, transform=eval_transform)
test_dataset = MultimodalDataset(test_df, DATA_DIR, word2idx, MAX_TEXT_LEN, transform=eval_transform, is_test=True)

num_workers = 2 if os.name != 'nt' else 0
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers)

print(f"Split: {len(train_dataset):,} train samples, {len(val_dataset):,} validation samples.")


In [ ]:
# ==============================================================================
# 5. DUAL-BRANCH MULTIMODAL MODEL ARCHITECTURE (< 15M Parameters)
# ==============================================================================
class VisionBranch(nn.Module):
    """Module 3: Lightweight CNN Backbone"""
    def __init__(self, backbone="mobilenet_v2"):
        super(VisionBranch, self).__init__()
        self.backbone = backbone.lower()
        if self.backbone == "mobilenet_v2":
            try:
                base = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
            except Exception:
                base = models.mobilenet_v2(weights=None)
            self.features = base.features
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.flatten = nn.Flatten()
            self.out_dim = 1280
        elif self.backbone == "resnet18":
            try:
                base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
            except Exception:
                base = models.resnet18(weights=None)
            self.features = nn.Sequential(
                base.conv1, base.bn1, base.relu, base.maxpool,
                base.layer1, base.layer2, base.layer3, base.layer4
            )
            self.pool = nn.AdaptiveAvgPool2d((1, 1))
            self.flatten = nn.Flatten()
            self.out_dim = 512
        else:
            raise ValueError(f"Unknown backbone: {backbone}")

    def forward(self, x):
        # x: [B, 3, H, W] -> out: [B, out_dim]
        return self.flatten(self.pool(self.features(x)))

class TextBranch(nn.Module):
    """Modules 4 & 5: Word Embeddings + Bidirectional GRU with Temporal Pooling"""
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super(TextBranch, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=2, batch_first=True,
            bidirectional=True, dropout=0.2
        )
        self.dropout = nn.Dropout(0.2)
        self.out_dim = hidden_dim * 2  # 256 features

    def forward(self, x):
        # x: [B, L] -> out: [B, 2*hidden_dim]
        emb = self.dropout(self.embedding(x))
        gru_out, _ = self.gru(emb)
        
        # Mask-aware average pooling over non-padded tokens
        mask = (x != 0).unsqueeze(-1).float()
        masked_out = gru_out * mask
        sum_pooled = masked_out.sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1.0)
        return sum_pooled / lengths

class DualBranchMultimodalModel(nn.Module):
    """End-to-End Dual-Branch Fusion Architecture"""
    def __init__(self, num_classes, vocab_size, backbone="mobilenet_v2"):
        super(DualBranchMultimodalModel, self).__init__()
        self.vision = VisionBranch(backbone=backbone)
        self.text = TextBranch(vocab_size=vocab_size, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM)
        
        # Late Fusion Concatenation
        fused_dim = self.vision.out_dim + self.text.out_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )

    def forward(self, images, texts):
        img_feat = self.vision(images)   # [B, vision_dim]
        text_feat = self.text(texts)     # [B, text_dim]
        fused = torch.cat((img_feat, text_feat), dim=1) # [B, fused_dim]
        return self.classifier(fused)   # [B, num_classes]

model = DualBranchMultimodalModel(
    num_classes=num_classes, vocab_size=len(word2idx), backbone=VISION_BACKBONE
).to(device)


In [ ]:
# ==============================================================================
# 6. PARAMETER LIMIT CHECK (< 15,000,000 PARAMETERS MANDATE)
# ==============================================================================
vision_params = sum(p.numel() for p in model.vision.parameters())
nlp_params = sum(p.numel() for p in model.text.parameters())
classifier_params = sum(p.numel() for p in model.classifier.parameters())
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 65)
print("      BCS714A MULTIMODAL MODEL PARAMETER AUDIT")
print("=" * 65)
print(f"  Vision Branch ({VISION_BACKBONE}) Parameters: {vision_params:>12,}")
print(f"  NLP Branch (Bi-GRU) Parameters:          {nlp_params:>12,}")
print(f"  Fusion Classifier Parameters:            {classifier_params:>12,}")
print("-" * 65)
print(f"  Total Parameters (All):                  {total_params:>12,}")
print(f"  Trainable Parameters:                    {trainable_params:>12,}")
print(f"  Hard Parameter Cap Limit:                {15_000_000:>12,}")
print(f"  Remaining Budget Headroom:               {(15_000_000 - total_params):>12,}")
print("=" * 65)

# MANDATORY ASSERTION
assert total_params < 15_000_000, f"VIOLATION: Model has {total_params:,} parameters, exceeding 15M limit!"
print(" [VERIFIED] Model strictly satisfies the < 15,000,000 parameter limit.\n")


In [ ]:
# ==============================================================================
# 7. BALANCED ACCURACY LOSS WEIGHTING & OPTIMIZER
# ==============================================================================
# Compute smoothed inverse class frequency weights
counts = train_split['category'].value_counts()
max_count = counts.max()
weights = np.ones(num_classes, dtype=np.float32)

for cat, idx in cat2idx.items():
    c = counts.get(cat, 1)
    weights[idx] = (max_count / max(c, 1)) ** 0.35

weights = weights / weights.mean()
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print("Class-weighted loss and Cosine Annealing optimizer configured.")


In [ ]:
# ==============================================================================
# 8. TRAINING LOOP WITH BALANCED ACCURACY TRACKING
# ==============================================================================
history = {"train_loss": [], "val_loss": [], "train_bacc": [], "val_bacc": []}
best_val_bacc = 0.0

print("Starting Multimodal Model Training...")
for epoch in range(1, EPOCHS + 1):
    # Train Pass
    model.train()
    running_loss = 0.0
    train_preds, train_targets = [], []
    
    pbar = tqdm(train_loader, desc=f"Epoch [{epoch}/{EPOCHS}] Train", leave=False)
    for images, texts, labels in pbar:
        images, texts, labels = images.to(device), texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images, texts)
        loss = criterion(outputs, labels)
        loss.backward()
        
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1).detach().cpu().numpy()
        train_preds.extend(preds)
        train_targets.extend(labels.detach().cpu().numpy())
        pbar.set_postfix(loss=loss.item())
        
    scheduler.step()
    train_loss = running_loss / len(train_dataset)
    train_bacc = balanced_accuracy_score(train_targets, train_preds)
    
    # Validation Pass
    model.eval()
    val_running_loss = 0.0
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for images, texts, labels in tqdm(val_loader, desc=f"Epoch [{epoch}/{EPOCHS}] Val", leave=False):
            images, texts, labels = images.to(device), texts.to(device), labels.to(device)
            outputs = model(images, texts)
            loss = criterion(outputs, labels)
            
            val_running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())
            
    val_loss = val_running_loss / len(val_dataset)
    val_bacc = balanced_accuracy_score(val_targets, val_preds)
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_bacc"].append(train_bacc)
    history["val_bacc"].append(val_bacc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] "
          f"| Train Loss: {train_loss:.4f} - Train BalAcc: {train_bacc*100:.2f}% "
          f"| Val Loss: {val_loss:.4f} - Val BalAcc: {val_bacc*100:.2f}%")
    
    if val_bacc > best_val_bacc:
        best_val_bacc = val_bacc
        torch.save(model.state_dict(), "best_multimodal_model.pth")
        print(f"  >>> New Best Model Saved! Val Balanced Accuracy: {best_val_bacc*100:.2f}%")


In [ ]:
# ==============================================================================
# 9. TRAINING & VALIDATION CURVES (Required for Technical Report)
# ==============================================================================
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss Curve
ax1.plot(epochs_range, history["train_loss"], 'o-', color='#1f77b4', linewidth=2.5, label='Train Loss')
ax1.plot(epochs_range, history["val_loss"], 's-', color='#d62728', linewidth=2.5, label='Val Loss')
ax1.set_title("Training vs Validation Loss", fontsize=14, fontweight='bold')
ax1.set_xlabel("Epoch", fontsize=12)
ax1.set_ylabel("Weighted Cross-Entropy Loss", fontsize=12)
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(fontsize=11)

# Balanced Accuracy Curve
ax2.plot(epochs_range, [acc * 100 for acc in history["train_bacc"]], 'o-', color='#2ca02c', linewidth=2.5, label='Train Balanced Acc (%)')
ax2.plot(epochs_range, [acc * 100 for acc in history["val_bacc"]], 's-', color='#ff7f0e', linewidth=2.5, label='Val Balanced Acc (%)')
ax2.set_title("Training vs Validation Balanced Accuracy", fontsize=14, fontweight='bold')
ax2.set_xlabel("Epoch", fontsize=12)
ax2.set_ylabel("Balanced Accuracy (%)", fontsize=12)
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(fontsize=11)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=300)
plt.show()
print("Training curves exported to 'training_curves.png'.")


In [ ]:
# ==============================================================================
# 10. INFERENCE & KAGGLE SUBMISSION EXPORT
# ==============================================================================
if os.path.exists("best_multimodal_model.pth"):
    model.load_state_dict(torch.load("best_multimodal_model.pth", map_location=device))
    print("Loaded best model checkpoint for final test inference.")

model.eval()
predictions = []
item_ids = []

print("\nGenerating predictions on Kaggle test set...")
with torch.no_grad():
    for images, texts, ids in tqdm(test_loader, desc="Inference"):
        images, texts = images.to(device), texts.to(device)
        outputs = model(images, texts)
        preds = outputs.argmax(dim=1).cpu().numpy()
        
        for p in preds:
            predictions.append(idx2cat[p])
        item_ids.extend(ids.numpy())

# Build DataFrame matching Kaggle sample_submission format
submission_df = pd.DataFrame({
    'id': item_ids,
    'category': predictions
})

# Format integrity validation
assert len(submission_df) == len(test_df), f"Row count mismatch: expected {len(test_df)}, got {len(submission_df)}"
assert list(submission_df.columns) == ['id', 'category'], "Columns must be ['id', 'category']"
assert submission_df['category'].isnull().sum() == 0, "Found NaN values in predictions!"

submission_df.to_csv("submission.csv", index=False)
print(f"\n[SUCCESS] 'submission.csv' generated ({len(submission_df):,} rows).")
print("Top predicted categories:")
print(submission_df['category'].value_counts().head(5))

from IPython.display import FileLink
FileLink('submission.csv')
